# Лабораторная работа №5: RAG для обнаружения дефектов металлических бутылок
**Дисциплина:** Искусственный интеллект  
**Студент:** Мыльников Александр Русланович  
**Группа:** ФИТ-221  
**Специальность:** 02.03.02 Фундаментальная информатика и информационные технологии  
**Тема диплома:** Разработка системы неразрушающего контроля для выявления дефектов металлических бутылок на конвейерной линии

## 1. Установка и импорт зависимостей

In [13]:
!pip install --upgrade pip -q langchain langchain-community langchain-chroma chromadb sentence-transformers pypdf unstructured python-dotenv markdown langchain-huggingface yandexcloud

import os
import sys
from pathlib import Path
from typing import List, Dict, Any
import logging
logging.basicConfig(level=logging.INFO)

from dotenv import load_dotenv
load_dotenv()
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# Добавим текущую папку в sys.path, чтобы импортировать модули
sys.path.append(str(Path.cwd()))


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Users\user\Desktop\univer_8\ml\ai-course-labs\.venv\Scripts\python.exe -m pip install --upgrade pip -q langchain langchain-community langchain-chroma chromadb sentence-transformers pypdf unstructured python-dotenv markdown langchain-huggingface yandexcloud
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


## 2. Создание документов (примеры для индексации)
Если документов ещё нет, создадим три файла, описывающих методы компьютерного зрения и дефектоскопию.

In [14]:
docs_dir = Path("./data/documents")
docs_dir.mkdir(parents=True, exist_ok=True)

doc1 = docs_dir / "computer_vision_basics.txt"
doc1.write_text("""
Основы компьютерного зрения для промышленного контроля

Компьютерное зрение (CV) – это область ИИ, позволяющая машинам «видеть».
На производстве CV применяется для автоматического обнаружения дефектов.

Ключевые задачи:
1. Классификация изображений – есть ли дефект.
2. Детекция объектов – локализация дефектов рамкой.
3. Сегментация – попиксельное выделение дефекта.

Метрики: Precision, Recall, mAP (mean Average Precision).
""", encoding="utf-8")

doc2 = docs_dir / "defect_detection_methods.md"
doc2.write_text("""
# Методы обнаружения дефектов металлических поверхностей

## Типы дефектов
- Царапины
- Вмятины
- Трещины

## Алгоритмы обработки
- **Otsu** – пороговая бинаризация для однородного фона.
- **Canny** – выделение границ царапин и трещин.
- **Морфология** – удаление шума.

## Нейросетевые подходы
- **YOLOv8** – детекция в реальном времени (до 100 FPS), mAP 0.92.
- **U-Net (ResNet34)** – сегментация дефектов, IoU > 0.85.
- **Vision Transformer (ViT)** – при большом объёме данных.

## Предобработка
Преобразование в оттенки серого, CLAHE, нормализация, медианная фильтрация.
""", encoding="utf-8")

doc3 = docs_dir / "production_line_inspection.txt"
doc3.write_text("""
Организация системы технического зрения на линии

Оборудование: камеры с глобальным затвором, кольцевое LED-освещение, GPU (NVIDIA Jetson).

Стадии обработки:
1. Захват кадра
2. Предобработка (оттенки серого, CLAHE)
3. Выделение ROI (области интереса)
4. Детекция дефектов (YOLO или U-Net)
5. Сигнал браковщику

Результаты испытаний: Recall 98.2%, Precision 96.5%, время обработки 25 мс/кадр.
""", encoding="utf-8")

print(f"Документы созданы в {docs_dir}")

Документы созданы в data\documents


## 3. Реализация и тестирование компонентов RAG

### 3.1 Загрузчик документов

In [17]:
from langchain_community.document_loaders import TextLoader, UnstructuredMarkdownLoader
from langchain_core.documents import Document
nltk.download('averaged_perceptron_tagger_eng')

class DocumentLoader:
    def __init__(self, source_directory: str):
        self.source_directory = Path(source_directory)
        self.supported_extensions = [".txt", ".md"]
        self.source_directory.mkdir(parents=True, exist_ok=True)

    def load_document(self, file_path: str) -> List[Document]:
        path = Path(file_path)
        ext = path.suffix.lower()
        if ext == ".txt":
            loader = TextLoader(file_path, encoding='utf-8')
        elif ext == ".md":
            loader = UnstructuredMarkdownLoader(file_path)
        else:
            return []
        docs = loader.load()
        for doc in docs:
            doc.metadata['source'] = str(file_path)
        return docs

    def load_directory(self) -> List[Document]:
        all_docs = []
        for ext in self.supported_extensions:
            for fpath in self.source_directory.glob(f"*{ext}"):
                all_docs.extend(self.load_document(str(fpath)))
        return all_docs

loader = DocumentLoader(str(docs_dir))
documents = loader.load_directory()
print(f"Загружено документов: {len(documents)}")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.
INFO:unstructured:Reading document from string ...
INFO:unstructured:Reading document ...
INFO:unstructured:HTML element instance has no attribute type
INFO:unstructured:HTML element instance has no attribute type
INFO:unstructured:HTML element instance has no attribute type
INFO:unstructured:HTML element instance has no attribute type
INFO:unstructured:HTML element instance has no attribute type
INFO:unstructured:HTML element instance has no attribute type


Загружено документов: 3


### 3.2 Разбиение на чанки

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"Получено чанков: {len(chunks)}")

Получено чанков: 4


### 3.3 Векторизация и хранение в ChromaDB

In [19]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Инициализация эмбеддинг-модели
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Создаём или пересоздаём коллекцию в ChromaDB
persist_dir = "./data/chroma_db"
!rm -rf {persist_dir}   # очищаем старые данные (для ноутбука)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_dir,
    collection_name="defect_collection"
)
print(f"Готово. Векторов: {vectorstore._collection.count()}")

INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2.
"rm" ­Ґ пў«пҐвбп ў­гваҐ­­Ґ© Ё«Ё ў­Ґи­Ґ©
Є®¬ ­¤®©, ЁбЇ®«­пҐ¬®© Їа®Ја ¬¬®© Ё«Ё Ї ЄҐв­л¬ д ©«®¬.
INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Готово. Векторов: 4


### 3.4 Поиск релевантных чанков (демонстрация)

In [20]:
query = "Как обнаружить царапины на металлической поверхности?"
retrieved = vectorstore.similarity_search_with_score(query, k=3)
for i, (doc, score) in enumerate(retrieved):
    print(f"\n[{i+1}] Сходство = {1-score:.4f}")
    print(f"Источник: {doc.metadata['source']}")
    print(doc.page_content[:200] + "...")


[1] Сходство = 0.2754
Источник: data\documents\defect_detection_methods.md
Методы обнаружения дефектов металлических поверхностей

Типы дефектов

Царапины

Вмятины

Трещины

Алгоритмы обработки

Otsu – пороговая бинаризация для однородного фона.

Canny – выделение границ цар...

[2] Сходство = -0.3661
Источник: data\documents\computer_vision_basics.txt
Основы компьютерного зрения для промышленного контроля

Компьютерное зрение (CV) – это область ИИ, позволяющая машинам «видеть».
На производстве CV применяется для автоматического обнаружения дефектов...

[3] Сходство = -0.4071
Источник: data\documents\defect_detection_methods.md
Предобработка

Преобразование в оттенки серого, CLAHE, нормализация, медианная фильтрация....


## 4. RAG-пайплайн с YandexGPT (или fallback)

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms import YandexGPT

class RAGPipeline:
    def __init__(self, vectorstore, llm=None, top_k=5):
        self.vectorstore = vectorstore
        self.llm = llm
        self.top_k = top_k
        self.prompt = ChatPromptTemplate.from_template("""
Ты — эксперт по компьютерному зрению для промышленного контроля.
Используй ТОЛЬКО предоставленный контекст для ответа.
Если ответа нет в контексте, скажи: "В документах нет информации".

Контекст:
{context}

Вопрос: {question}
Ответ:
""")

    def _format_context(self, docs_with_scores):
        context = ""
        for i, (doc, score) in enumerate(docs_with_scores):
            context += f"[Источник {i+1}]\n{doc.page_content}\n\n"
        return context

    def query(self, question: str):
        docs_with_scores = self.vectorstore.similarity_search_with_score(question, k=self.top_k)
        if not docs_with_scores:
            return "Нет релевантных документов.", []
        context = self._format_context(docs_with_scores)
        if self.llm:
            prompt_text = self.prompt.format(context=context, question=question)
            answer = self.llm.invoke(prompt_text)
        else:
            answer = f"Найдено {len(docs_with_scores)} фрагментов.\n\n{context}"
        return answer, docs_with_scores

# Инициализация LLM
iam_token = os.getenv("YANDEX_IAM_TOKEN")
folder_id = os.getenv("YANDEX_FOLDER_ID")
llm = None
if iam_token and folder_id:
    llm = YandexGPT(iam_token=iam_token, folder_id=folder_id, temperature=0.3, max_tokens=500)
    print("YandexGPT подключён")
else:
    print("Работаем без LLM (только поиск)")

rag = RAGPipeline(vectorstore, llm=llm, top_k=3)

YandexGPT подключён


## 5. Тестирование на вопросах по теме диплома

In [22]:
test_questions = [
    "Как обнаружить царапины на металлической поверхности с помощью компьютерного зрения?",
    "Какие нейросетевые архитектуры лучше всего подходят для сегментации дефектов на металлических бутылках?",
    "Как предобработать изображения для улучшения качества обнаружения вмятин?"
]

for q in test_questions:
    print("\n" + "="*70)
    print(f"Вопрос: {q}")
    answer, sources = rag.query(q)
    print(f"Ответ:\n{answer}\n")
    print("Источники:")
    for i, (doc, score) in enumerate(sources):
        print(f"  {i+1}. {doc.metadata['source']} (score={1-score:.3f})")
    print("="*70)


Вопрос: Как обнаружить царапины на металлической поверхности с помощью компьютерного зрения?
Ответ:
Для обнаружения царапин на металлической поверхности с помощью компьютерного зрения можно использовать следующие алгоритмы обработки:

1. **Otsu** — пороговая бинаризация для однородного фона.
2. **Canny** — выделение границ царапин и трещин.
3. **Морфология** — удаление шума.

Также можно использовать нейросетевые подходы, например, **YOLOv8** для детекции в реальном времени (до 100 FPS) или **U-Net (ResNet34)** для сегментации дефектов.

Источники:
  1. data\documents\defect_detection_methods.md (score=0.331)
  2. data\documents\computer_vision_basics.txt (score=-0.017)
  3. data\documents\defect_detection_methods.md (score=-0.383)

Вопрос: Какие нейросетевые архитектуры лучше всего подходят для сегментации дефектов на металлических бутылках?
Ответ:
U-Net (ResNet34) – сегментация дефектов, IoU > 0.85.

Источники:
  1. data\documents\defect_detection_methods.md (score=0.301)
  2. data\

## 6. Специализированный промпт для обнаружения дефектов (дополнительно)
Класс `DefectDetectionRAGPipeline` с расширенными метаданными (как в лабораторной).

In [23]:
class DefectDetectionRAGPipeline(RAGPipeline):
    def __init__(self, vectorstore, llm=None, top_k=5):
        super().__init__(vectorstore, llm, top_k)
        self.prompt = ChatPromptTemplate.from_template("""
Ты — эксперт по компьютерному зрению для промышленности.
Используй ТОЛЬКО контекст. Если вопрос о дефекте, укажи тип дефекта, алгоритмы, метрики.

Контекст:
{context}

Вопрос: {question}
Ответ (с указанием методов CV):
""")

    def query(self, question: str):
        answer, sources = super().query(question)
        # дополнительно можно извлечь упомянутые методы
        return answer, sources

# Демонстрация
defect_rag = DefectDetectionRAGPipeline(vectorstore, llm=llm, top_k=3)
q = "Какой алгоритм лучше для обнаружения вмятин на металле?"
print("\nСпециализированный RAG:")
ans, src = defect_rag.query(q)
print(f"Вопрос: {q}\nОтвет:\n{ans}")


Специализированный RAG:
Вопрос: Какой алгоритм лучше для обнаружения вмятин на металле?
Ответ:
Для обнаружения вмятин на металле можно использовать алгоритм Canny. Этот алгоритм предназначен для выделения границ объектов на изображении, что позволяет эффективно обнаруживать такие дефекты, как вмятины. Также может быть полезна предобработка изображения с помощью морфологических операций для удаления шума и улучшения видимости границ дефектов.
